In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import time


from __future__ import annotations
from yandex_cloud_ml_sdk import YCloudML


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [2]:
X_test = pd.read_csv('ver_1/X_test.csv',index_col=0)

In [3]:
X_test.head(10)

,purpose
222196,Благотворительное пожертвование на уставную деятельность. НДС не облагается
474146,БЛАГОТВОРИТЕЛЬНЫЙ ВЗНОС ЗА 13/04/2025;Добровольное пожертвование САВЕЛЬЕВ ДМИТРИЙ АЛЕКСАНДРОВИЧ;semenixina.mar@yandex.ru
233469,"Перевод средств по договору б/н от 23.07.2020 по Реестру Операций от 25.07.2024. Сумма комиссии 8 руб. 70 коп., НДС не облагается."
7601,"Оплата ветеринарных услуг по счету 11369 от 14.03.2023 (Ярулина, соб Лохматая), НДС не облагается"
436337,Добровольное пожертвование 10.00
282842,Благотворительное пожертвование на уставную деятельность. НДС не облагается
338292,(283.0703.0120165300.635 л/с 02133D07240) ДК 24В26 л/с 03283D10001 БО 213D100024000012 Субсид на иные цели (фин обеспеч испол мун соц заказа на оказан мун услуг) по соглаш 13 от 09.01.2024 пр1139 от 12.11.2024
560686,Благотворительное пожертвование на уставную деятельность. НДС не облагается
393765,Благотворительное пожертвование на уставную деятельность. НДС не облагается
207682,Благотворительное пожертвование на уставную деятельность. НДС не облагается


In [4]:
y_pred_ygpt_ft = pd.DataFrame(columns=["universal_category"])

sdk = YCloudML(
        folder_id="YOUR_FOLDER_ID",
        auth="YOUR_YANDEX_CLOUD_TOKEN",
    )

model = sdk.models.text_classifiers(
        "cls://YOUR_FOLDER_ID/yandexgpt-lite/rc@tamrqvji6378ld14foam4"
    )

for index__, text in tqdm(X_test['purpose'].items(), total=len(X_test)):
    attempts = 0
    max_attempts = 3
    while attempts < max_attempts:
        try:
            result = model.run(str(text))  # приводим текст к строке
            best_label = max(result, key=lambda x: x.confidence).label
            y_pred_ygpt_ft.loc[index__, "universal_category"] = best_label
            break
        except Exception as e:
            attempts += 1
            print(f"Ошибка на index {index__}, попытка {attempts}/{max_attempts}: {e}")
            time.sleep(2)
    else:
        # если все попытки неудачные, присваиваем NaN
        y_pred_ygpt_ft.loc[index__, "universal_category"] = np.nan

        
y_pred_ygpt_ft.to_csv("y_pred_ygpt_ft.csv", index=True, header=["universal_category"])
    

100%|██████████| 5000/5000 [2:51:19<00:00,  2.06s/it]  


In [5]:
y_pred_ygpt_ft.to_csv("y_pred_ygpt_ft.csv", index=True, header=["universal_category"])